# `00_bootstrap_helpers` — confirm the AIDP connectors helper package is set up

Run this notebook **once per AIDP workspace** after the plugin's helpers have been uploaded to `/Workspace/Shared/oracle_ai_data_platform_connectors/scripts/`.

If you haven't uploaded the helpers yet, ask Claude: *"set up the AIDP connectors plugin in this workspace"* — the `aidp-connectors-bootstrap` skill drives the upload via MCP. Or upload `scripts/oracle_ai_data_platform_connectors/` manually via the AIDP UI to that path.

**Pass criteria:** the final cell prints `BOOTSTRAP OK` plus the package version and a list of submodules.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
# Confirm the directory layout the helpers expect.
import os, pathlib
expected = pathlib.Path('/Workspace/Shared/oracle_ai_data_platform_connectors/scripts/oracle_ai_data_platform_connectors')
if not expected.exists():
    raise RuntimeError(
        f'Helpers not found at {expected}. Run the aidp-connectors-bootstrap skill (ask Claude: "set up the AIDP connectors plugin") '
        f'or upload the plugin scripts/ directory to /Workspace/Shared/ manually.'
    )
files = sorted(p.name for p in expected.rglob('*.py'))
print(f'found {len(files)} Python files under {expected}')
for f in files: print(' ', f)


In [ ]:
# Sanity-import every public submodule.
import importlib
import oracle_ai_data_platform_connectors as pkg
from oracle_ai_data_platform_connectors import auth, jdbc, rest, streaming
from oracle_ai_data_platform_connectors.auth import (
    write_wallet_to_tmp, generate_db_token, from_inline_pem,
    http_basic_session, oauth_token, get_secret,
)
from oracle_ai_data_platform_connectors.jdbc import (
    build_oracle_jdbc_url,
    spark_jdbc_options_wallet, spark_jdbc_options_dbtoken, spark_jdbc_options_password,
)
from oracle_ai_data_platform_connectors.rest import fusion, epm, essbase  # noqa: F401
from oracle_ai_data_platform_connectors.streaming import (
    bootstrap_for_region, build_kafka_options_sasl_plain, validate_checkpoint_path,
)
print('all imports OK; package version:', pkg.__version__)


In [ ]:
# Quick logic smoke test: run the URL builder and the checkpoint validator.
url = build_oracle_jdbc_url(tns_alias='atp_high', tns_admin='/tmp/wallet/atp')
assert url == 'jdbc:oracle:thin:@atp_high?TNS_ADMIN=/tmp/wallet/atp', f'unexpected URL: {url}'
import pytest as _pytest
try:
    validate_checkpoint_path('/Workspace/cp')
    raise AssertionError('checkpoint validator should have raised')
except ValueError:
    pass
print('smoke test OK')


In [ ]:
# Final result marker — the live-test driver picks this up if you run it as part of a batch.
import json, time
summary = {
    'connector': 'bootstrap',
    'auth': 'n/a',
    'rows': 1,                  # 1 = bootstrap success
    'schema': ['BOOTSTRAP_OK'],
    'package_version': pkg.__version__,
    'timestamp_utc': int(time.time()),
}
print('BOOTSTRAP OK')
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
